# Stretch: more cleaning, more charts, worked solutions

The answers to the stretch notebook. Try each one before looking.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy" / "plays_messy.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

sys.path.insert(0, "sessions/session-09/solutions")
from cleaning import clean_plays, load_messy

raw = load_messy("data/messy/plays_messy.csv")
plays = clean_plays(raw)
clean = pd.read_csv("data/clean/plays.csv")

os.makedirs("output", exist_ok=True)
print(len(raw), len(plays), len(clean))

## Stretch 1: a damage report

In [ ]:
def damage_report(df):
    """Return one row per column: blanks, distinct values, and distinct values after strip and lower."""
    rows = []
    for column in df.columns:
        text = df[column].astype(str)
        rows.append({
            "column": column,
            "blanks": (text == "").sum(),
            "spellings": text.nunique(),
            "after strip and lower": text.str.strip().str.lower().nunique(),
        })
    return pd.DataFrame(rows)


report = damage_report(raw)

assert len(report) == 9
assert report.set_index("column").loc["device", "spellings"] == 21
assert report.set_index("column").loc["device", "after strip and lower"] == 6
assert report.set_index("column").loc["genre", "blanks"] == 36

report

A list of dictionaries, one per column, becomes a DataFrame in one call.
The keys become the column names. That is the session 4 shape, and now you
have built a table with it instead of reading one.

Reading the report:

* **the most spellings is `play_id`, with 2,183**, and that is not damage
  at all. An id is supposed to be different on every row. A number of
  spellings only means something next to what you expected
* **genre and device** drop sharply after strip and lower: 33 to 9, and 21
  to 6. That gap is the damage. (9 and 6, not 8 and 5, because the blank
  counts as a value.)
* **`minutes played `** barely changes, because the problem there is not
  capitals. The report does not catch `" min"`. No single check catches
  everything, which is why notebook 01 looked at each column by eye too
* **country has no blanks**. On the default read in notebook 01,
  `isna().sum()` said 49 missing. Those were the `"NA"` rows, which this
  report sees as an ordinary two-letter value, because `load_messy` keeps
  text as text

## Stretch 2: draw the wrong dates

In [ ]:
deduped = raw.drop_duplicates().sort_values("play_id")
guessed = pd.to_datetime(deduped["played_at"], format="mixed").dt.strftime("%Y-%m-%d")
guessed_months = guessed.str[:7]

right = plays["played_at"].str[:7].value_counts().sort_index()
wrong = guessed_months.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(right.index, right.values, marker="o", color="#4f6ddb", label="each format on purpose")
ax.plot(wrong.index, wrong.values, marker="o", color="#c0392b", label='format="mixed"')
ax.set_title("A blanket date parse moves 158 plays to the wrong month")
ax.set_ylabel("plays per month")
ax.set_ylim(bottom=0)
ax.tick_params(axis="x", labelrotation=90)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
moved = (wrong - right).sort_values()
print(moved.head(2))
print(moved.tail(2))

assert (guessed.values != plays["played_at"].values).sum() == 158      # both sorted by play_id
assert moved["2025-01"] == -8 and moved["2025-12"] == 7

158 here rather than 159, because one of the 159 was a duplicate row.

January 2025 lost 8 plays and December 2025 gained 7. That is what a
day/month swap does: the 12th of January becomes the 1st of December, and
so on. Plays are not lost, they are moved, so the total for the year stays
right and every monthly number is a little bit off.

**Could you spot it from the wrong line alone?** No. It still dips every
summer, it still peaks in autumn. Nothing about it looks broken. That is
the whole danger of a silent wrong parse, and why checking each format is
the only defence.

Two lines means a legend. With one line, the title does that job.

## Stretch 3: what were the unknowns?

In [ ]:
merged = plays.merge(clean, on="play_id", suffixes=("_ours", "_clean"), validate="one_to_one")
assert len(merged) == 2183

unknowns = merged[merged["device_ours"] == "unknown"]
assert len(unknowns) == 69

comparison = pd.DataFrame({
    "unknowns were": unknowns["device_clean"].value_counts(),
    "share of unknowns": unknowns["device_clean"].value_counts(normalize=True).round(3),
    "share of all plays": clean["device"].value_counts(normalize=True).round(3),
})
comparison

35 phones, 14 laptops, 10 speakers, 6 cars, 4 tablets. Phones are 51% of
the unknowns and 48% of all plays; every device is within a few points of
its share. The blanks hit every device about evenly.

**Why that matters.** If the blanks are spread evenly, dropping them or
labelling them barely changes each device's share, so plays per device
stays fair either way. If the export had blanked out, say, mostly car
plays, then dropping the blanks would quietly make the car look less used
than it was. Missing at random and missing for a reason need different
treatment, and you can only tell them apart by looking.

In real work you have no clean file to merge with. You can still compare
the rows with blanks against the rest on columns you **do** have: their
genres, their dates, their skip rate.

In [ ]:
print(f"skip rate of the unknowns: {unknowns['skipped_ours'].mean():.1%}")
print(f"skip rate of everything:   {plays['skipped'].mean():.1%}")

15.9% against 11.8%. That is 11 skips out of 69, where the overall rate
would predict about 8. With a group this small, three extra skips is the
kind of difference chance produces all the time, which is one more reason
not to rank `"unknown"` against the real devices.

## Stretch 4: an honest skip-rate chart

In [ ]:
rates = plays.groupby("device")["skipped"].agg(plays="size", rate="mean")
rates["rate"] = rates["rate"] * 100
rates = rates.sort_values("rate")                 # barh draws from the bottom up

colours = ["#9aa3ad" if device == "unknown" else "#4f6ddb" for device in rates.index]
labels = [f"{rate:.1f}% of {n}" for rate, n in zip(rates["rate"], rates["plays"])]

known = rates.drop("unknown")
top_device = known.index[-1]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(rates.index, rates["rate"], color=colours)
ax.bar_label(bars, labels=labels, padding=4)
ax.set_title(f"Speaker plays are skipped most: {known['rate'].iloc[-1]:.1f}%")
ax.set_xlabel("plays skipped (%)")
ax.set_xlim(0, 20)
ax.spines[["top", "right"]].set_visible(False)

fig.savefig("output/skip_rate_by_device.png", dpi=150, bbox_inches="tight")
plt.show()

assert top_device == "speaker"

The grey bar is still there, at the top, longest of all, and it does not
win, because the title is computed from the real devices only.

`zip` walks two lists side by side, one pair at a time, which is what you
need to put two numbers in each label. Labelling each bar with its play
count lets the reader judge for themselves how much to trust each rate:
69 plays behind a bar should be read differently from 1,004.

## Stretch 5: two years, side by side

In [ ]:
genre_year = plays.groupby([plays["played_at"].str[:4], "genre"]).size().unstack(0)
genre_year = genre_year.sort_values("2024", ascending=False)

assert genre_year.loc["Folk", "2024"] - genre_year.loc["Folk", "2025"] == 34
genre_year

`unstack(0)` takes the first level of the group, the year, and turns it into
columns. It is the pivot table from session 4 stretch 5, in one line.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

left.bar(genre_year.index, genre_year["2024"], color="#4f6ddb")
left.set_title("2024")
left.set_ylabel("plays")
left.tick_params(axis="x", labelrotation=45)

right.bar(genre_year.index, genre_year["2025"], color="#4f6ddb")
right.set_title("2025")
right.tick_params(axis="x", labelrotation=45)

fig.suptitle("Much the same mix in both years; Folk fell most, from 236 to 202 plays")
plt.show()

`fig.suptitle` is a title for the whole figure, above the two chart
titles.

**Without `sharey=True`**, each chart picks its own y-axis. The 2025 chart
tops out lower, so matplotlib stretches it, and its bars end up about as
tall as 2024's. A reader comparing the two would conclude nothing changed,
and they would be reading the scale, not the data. Side-by-side charts
that are meant to be compared must share an axis.

## Stretch 6: guess the missing minutes, and get marked

In [ ]:
TRUTH = clean["minutes_played"].sum()
minutes = plays["minutes_played"]

by_artist = plays.groupby("artist_name")["minutes_played"].transform("median")
by_artist_and_skip = plays.groupby(["artist_name", "skipped"])["minutes_played"].transform("median")


def imputed_total(filled):
    """Return the total of a minutes column after its gaps have been filled."""
    return filled.sum()


methods = {
    "leave missing": minutes,
    "average of all plays": minutes.fillna(minutes.mean()),
    "artist median": minutes.fillna(by_artist),
    "artist and skipped median": minutes.fillna(by_artist_and_skip),
}

results = pd.DataFrame([
    {"method": name, "total": round(imputed_total(filled), 1),
     "off by": round(imputed_total(filled) - TRUTH, 1)}
    for name, filled in methods.items()
])

assert round(minutes.sum(), 1) == 8608.0
assert round(TRUTH, 1) == 8980.4
assert by_artist_and_skip[minutes.isna()].notna().all()     # every gap got a guess
results

**1. Why is the artist median worse than the plain average?** Because of
who the missing plays are.

In [ ]:
print(f"skipped, among the missing:  {plays.loc[minutes.isna(), 'skipped'].mean():.1%}")
print(f"skipped, among all plays:    {plays['skipped'].mean():.1%}")

16.0% of the missing plays were skipped, against 11.8% overall, and a
skipped play is short. The artist median is the length of a typical
**full** play by that artist, so for every skipped play it guesses several
minutes too many. Using more information made it worse, because it was the
wrong information. Adding `skipped` to the group fixes exactly that, and
lands within 3.4 minutes of the truth.

**2. Is leaving them missing still the most honest?** It has the biggest
error, 372.4 minutes short, and it is the only total that is a fact: it is
the exact total of the 2,089 plays whose length we know. Every other row
is an estimate. So the label is what matters:

* "8,608.0 minutes, from the 2,089 plays with a recorded length" is true
* "about 8,977 minutes, with 94 play lengths estimated" is honest
* "8,977.0 minutes", with no note, is a guess dressed up as a measurement

**3. With no clean file, how do you choose?** You check the assumption
behind each method against the data you have, as question 1 did: are the
missing rows like the others? Here they were skipped more often, which
rules out any method that ignores `skipped`. And you say what you did.
An estimate that is labelled is a tool; one that is not is a mistake
waiting for someone to find it.